# COS30049 A2 - Round 2: continued pretraining of the 152M model on both T4s

Continues round 1's 152M model (validation loss 3.1305 after 2.0B tokens) on new FineWeb-Edu tokens
(`configs/text_150m_pretrain_cont.json`, `docs/KAGGLE_LM_PLAN.md`), with a warm-up, stable, decay schedule, on both
GPUs of a T4 x2 session (torchrun, `lm/dist.py`). Pretraining is one chain of sessions: each resumes from the
previous one's checkpoint.

**Settings:** Accelerator **GPU T4 x2**, **Internet on**. **Inputs:** this account's dataset
`cos30049a2-text150m-base` and notebook `cos30049a2-text150m-data`; from the second session on, also this notebook's
own previous output, with `RESUME_FROM` set below. Then **Save Version -> Save & Run All (Commit)**.

**Session 1** checks that the round-1 weights give the same validation loss in fp16 as in bf16 (T4s have no bf16;
within `FP16_TOLERANCE` of 3.1305), benchmarks micro-batch sizes on both GPUs, and sets `total_tokens` by the gate:
the tokens the measured speed can train in `GPU_HOURS` (docs/KAGGLE_LM_PLAN.md section 6), saved in `gate.json` so
later sessions keep it. **Every session** then trains until the session's time is nearly up, saving a checkpoint
every 30 minutes. To train for longer (another account's quota), raise `TOTAL_TOKENS` before the decay starts (the
last 15%).

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "1aab9817dee360269b9bf9a8f7b142cddaa0c8cb"   # the code this run uses; recorded in kaggle_run.json
SESSION_HOURS = 11.75  # Kaggle ends a GPU session at 12 h
GPU_HOURS = 50.0       # the gate: GPU hours this chain may use in all: this week's quota and the next (reset Sat
                       # 07:00 GMT+7) on this account, less about 6 h kept for the chat and explanation stages
TOTAL_TOKENS = None    # None: session 1's gate decides; a number overrides gate.json (only before the decay starts)
ROUND1_VAL_LOSS = 3.1305  # runs/text-150m/log.jsonl, last evaluation (bf16 on the RTX 3060, the same 160 windows)
FP16_TOLERANCE = 0.01  # docs/KAGGLE_LM_PLAN.md section 5, the fp16 check
RESUME_FROM = None     # e.g. "/kaggle/input/<this notebook>/runs/text-150m-cont" from the second session on
PINS = ["tokenizers==0.21.1", "numpy==2.2.6"]  # environment.yml; torch is Kaggle's own (recorded)

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if shutil.which("nvidia-smi") is None:
    raise SystemExit("No GPU in this session: Settings panel -> Accelerator -> GPU T4 x2 (needs a verified phone).")
GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
RUN = WORK / "runs" / "text-150m-cont"

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

def find_input(rel):
    """The file under /kaggle/input whose path ends with rel (inputs mount under their own folder names)."""
    hits = [p for p in Path("/kaggle/input").rglob(Path(rel).name) if p.as_posix().endswith(rel)]
    if not hits:
        raise SystemExit(f"{rel} is not in any input: add the inputs named at the top")
    return hits[0]


def check_base():
    """The round-1 files (dataset cos30049a2-text150m-base), each checked against the dataset's SHA256SUMS.txt."""
    sums = find_input("SHA256SUMS.txt")
    for line in sums.read_text().splitlines():
        digest, name = line.split(maxsplit=1)
        assert sha256(sums.parent / name.lstrip("*")) == digest, f"{name} differs from SHA256SUMS.txt"
    return sums.parent

sh("nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv")
TORCHRUN = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", str(GPUS)]

In [ ]:
base = check_base()
data_manifest = json.loads(find_input("data_manifest.json").read_text(encoding="utf-8"))
DATA = find_input("text_cont/meta.json").parent
for rel, digest in data_manifest["files"].items():
    if not rel.endswith("train.bin"):  # 6 GB: its size is checked instead of its hash, which would take minutes
        assert sha256(DATA.parent / rel) == digest, f"{rel} differs from data_manifest.json"
assert (DATA / "train.bin").stat().st_size == 2 * data_manifest["train_tokens"], "train.bin is incomplete"
if RESUME_FROM and not (RUN / "ckpt_last.pt").exists():
    shutil.copytree(RESUME_FROM, RUN, dirs_exist_ok=True)
    print("resuming from", RESUME_FROM)
RUN.mkdir(parents=True, exist_ok=True)


def write_config(**override):
    cfg = json.loads((SRC / "configs" / "text_150m_pretrain_cont.json").read_text(encoding="utf-8"))
    cfg.update({"data_dir": str(DATA), "out_dir": str(RUN), "init_from": str(base / "text150m_base.pt"), **override})
    path = Path("/tmp/text150m_cont.json")
    path.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    return path

In [ ]:
gate_path = RUN / "gate.json"
if not gate_path.exists():
    # The fp16 check: round 1 trained in bf16, which T4s lack; fp16 overflows sooner, so the starting point must match.
    sh(*TORCHRUN, "-m", "lm.train", "--config", str(write_config()), "--eval-only", log="fp16-check.log")
    fp16 = json.loads((RUN / "eval_only.json").read_text(encoding="utf-8"))["val_loss"]
    print(f"fp16 validation loss {fp16:.4f} against round 1's {ROUND1_VAL_LOSS} (bf16)")
    if abs(fp16 - ROUND1_VAL_LOSS) > FP16_TOLERANCE:
        raise SystemExit("fp16 check failed: do not train in fp16 (docs/KAGGLE_LM_PLAN.md section 7)")
    sh(*TORCHRUN, "-m", "lm.train", "--config", str(write_config()), "--bench", log="bench.log")
    bench = json.loads((RUN / "bench.json").read_text(encoding="utf-8"))
    rate = bench["tok_per_s_per_gpu"] * GPUS
    # The gate (plan section 6): what the measured speed trains in GPU_HOURS, less 10% for evaluations and handoffs,
    # in whole optimizer steps, and no more than the tokens prepared.
    batch = json.loads((SRC / "configs" / "text_150m_pretrain_cont.json").read_text(encoding="utf-8"))["batch_tokens"]
    total = min(int(rate * GPU_HOURS * 3600 * 0.9), data_manifest["train_tokens"]) // batch * batch
    gate_path.write_text(json.dumps({"micro_batch": bench["micro_batch"], "tok_per_s": rate, "gpus": GPUS,
                                     "gpu_hours": GPU_HOURS, "total_tokens": total, "fp16_val_loss": fp16},
                                    indent=2) + "\n", encoding="utf-8")
gate = json.loads(gate_path.read_text(encoding="utf-8"))
if TOTAL_TOKENS:
    gate["total_tokens"] = TOTAL_TOKENS
    gate_path.write_text(json.dumps(gate, indent=2) + "\n", encoding="utf-8")
print(json.dumps(gate, indent=2))

In [ ]:
config = write_config(micro_batch=gate["micro_batch"], total_tokens=gate["total_tokens"])
hours = SESSION_HOURS - (time.time() - T0) / 3600 - 0.3  # a margin for the final checkpoint and evaluation
sh(*TORCHRUN, "-m", "lm.train", "--config", str(config), "--max-hours", f"{hours:.2f}", log="train.log")

In [ ]:
# What ran, where, with which versions; and the run folder's size (the output keeps at most 20 GB).
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
last = [json.loads(l) for l in (RUN / "log.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
evals = [r for r in last if "val_loss" in r]
(WORK / "kaggle_run.json").write_text(json.dumps({
    "commit": COMMIT, "data_commit": data_manifest["commit"], "gpus": gpu, "python": sys.version.split()[0],
    "packages": {p: version(p) for p in ("torch", "numpy", "tokenizers")}, "gate": gate, "resumed_from": RESUME_FROM,
    "session_hours": round((time.time() - T0) / 3600, 2), "last_step": max(r.get("step", 0) for r in last),
    "last_eval": evals[-1] if evals else None, "finished": (RUN / "model.pt").exists(),
}, indent=2), encoding="utf-8")
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 2**20:9.1f} MB  {p.relative_to(WORK)}")